# Tratativa dos cartões
## Etapa tratativa dos dados e gravação na silver

Esta etapa consiste em tratar os dados dos cartões, renomear colunas, alterar datatypes e selecionar dados relevantes para o processo.
Alguns jogos (ao menos 10) estão com o array de cartões vazio.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_silver = spark.read.table("apifootball.silver.cards") #não existe na primeira execução
max_dh_bronze = (
    spark.read.table("apifootball.bronze.matches_raw")
    .agg(max("dh_processing_br").alias("max_dh"))
    .collect()[0]["max_dh"]
)
df_bronze = (
    spark.read.table("apifootball.bronze.matches_raw").alias("r")
    .filter(col("r.match_status") == "Finished")
    .filter(col("r.dh_processing_br") == max_dh_bronze)
    .join(df_silver.alias("s"), col("r.match_id") == col("s.match_id"), "leftanti") # segunda rodada +, primeira comentar
)#167

In [0]:
df_cards_exploded = (
    df_bronze
    .select(
        col("match_id").cast("int").alias("match_id"),
        col("match_hometeam_id").cast("int"),
        col("match_hometeam_name"),
        col("match_awayteam_id").cast("int"),
        col("match_awayteam_name"),
        explode("cards").alias("card_event"),
        col("dh_processing_br").alias("datetime_processing_bronze_br"),
    )
)

In [0]:
df_cards = (
    df_cards_exploded
    .select(
        col("match_id"),
        col("card_event.time").alias("minute"),
        col("card_event.card").alias("card_type"),        
        when(
            col("card_event.home_fault") != "",
            col("card_event.home_fault")
        ).otherwise(
            col("card_event.away_fault")
        ).alias("player_name"),
        when(
            col("card_event.home_fault") != "",
            col("match_hometeam_id")
        ).otherwise(
            col("match_awayteam_id")
        ).alias("team_id"),        
        when(
            col("card_event.home_fault") != "",
            col("match_hometeam_name")
        ).otherwise(
            col("match_awayteam_name")
        ).alias("team_name"),
        when(
            col("card_event.home_fault") != "",
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("is_hometeam"),
        when(
            col("card_event.away_fault") != "",
            lit(1)
        ).otherwise(
            lit(0)
        ).alias("is_awayteam"),   
        expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br'),
        col('datetime_processing_bronze_br')
    )        
)

In [0]:
df_cards_filtered = (
    df_cards
    .filter(col("card_type").isNotNull())
).orderBy(col("match_id"))

In [0]:
df_cards_filtered.write.mode("append").saveAsTable("apifootball.silver.cards")

In [0]:
#spark.read.table("apifootball.silver.cards").display()